# Swing Structure & BOS Analysis — v2

## Logic Summary

### Swing Detection
- 3-bar rule: Swing High at bar[i] if `high[i] > high[i-1] AND high[i] > high[i+1]`
- Confirmed at bar[i+1] close. One-bar lag, no lookahead.

### Major vs Minor Classification
- A swing low is **Minor** until the Major Swing High above it is broken via BOS.
- On confirmed bullish BOS: the **lowest swing low of the entire leg** (from prev major SL to BOS bar) is retroactively promoted to **Major Swing Low**.
- The **first confirmed 3-bar swing high that forms after the BOS bar** becomes the new **Major Swing High**.
- Symmetrical for bearish side.

### BOS Rules
- **Trigger**: candle **closes** beyond the Major Swing level.
- **Buffer (4 bars)**: after trigger, if price returns back through the broken level within 4 bars → `FAILED_BOS`. Level stays intact, leg continues.
- **Confirmed BOS**: price holds beyond the level for 4 bars → state transitions, major levels promoted.
- **Scoring** (analytical only, does not affect confirmation):
  - `wick_score`: how far the wick extended beyond the level
  - `close_type`: WICK / CLOSE / BODY (close position relative to level)
  - `disp_score`: weighted body ratio + range expansion + relative volume
  - `total_score`: disp_score + close_bonus

### Range Detection
- If no BOS fires within `RANGE_BARS` consecutive bars, the market is tagged RANGING.
- Range does not block BOS detection — it is an informational label only.

**Run cells top to bottom. Edit paths and parameters in Cell 2 only.**

## Cell 1 — Installs & Imports

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scipy', 'seaborn', '-q'])

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import spearmanr, mannwhitneyu
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 40)
plt.style.use('dark_background')
np.random.seed(42)
print('Imports complete.')

## Cell 2 — Master Config
**All paths and parameters here. Change nothing elsewhere.**

In [ ]:
CFG = {

    # ── Paths ──────────────────────────────────────────────────────────────
    'PATH_KLINES_1M'     : '/content/BTCUSDT-1m-2021-010.csv',
    'PATH_AGGTRADES'     : None,   # optional aggTrades CSV, set None to skip

    # ── Timeframe for structure detection ──────────────────────────────────
    # Binance resample rule: '1min','5min','15min','1h','4h','1D'
    'STRUCTURE_TF'       : '1D',

    # ── Data window ────────────────────────────────────────────────────────
    'DAYS_TO_LOAD'       : 365,

    # ── BOS Buffer ─────────────────────────────────────────────────────────
    # After a BOS trigger (close beyond level), price must NOT return through
    # the level within this many bars. If it does → FAILED_BOS.
    'BOS_BUFFER_BARS'    : 4,

    # ── Range Detection ────────────────────────────────────────────────────
    # If no confirmed BOS fires within this many bars, tag as RANGING.
    # Informational only — does not block BOS detection.
    'RANGE_BARS'         : 20,

    # ── BOS Scoring Parameters ─────────────────────────────────────────────
    # Weights for displacement score — must sum to 1.0
    'W_BODY'             : 0.35,   # candle body / total range
    'W_RANGE'            : 0.35,   # bar range / ATR(14), normalised 0-1
    'W_VOL'              : 0.30,   # volume / vol_MA(20), normalised 0-1

    # Normalisation caps
    'RANGE_EXP_CAP'      : 3.0,
    'VOL_REL_CAP'        : 3.0,

    # Close position bonus (added on top of displacement score)
    # WICK  = high/low crossed level but close did not
    # CLOSE = close crossed level but open did not
    # BODY  = both open AND close beyond level
    'CLOSE_BONUS_WICK'   : 0.00,
    'CLOSE_BONUS_CLOSE'  : 0.20,
    'CLOSE_BONUS_BODY'   : 0.30,

    # ── ATR / Volume windows ───────────────────────────────────────────────
    'ATR_PERIOD'         : 14,
    'VOL_MA_PERIOD'      : 20,

    # ── Outcome Measurement ────────────────────────────────────────────────
    'BOS_FWD_BARS'       : [6, 12, 24, 48],
    # Genuine = price stays on correct side for this many bars post-BOS
    'BOS_GENUINE_HOLD'   : 12,
    'BOS_HOLD_BUFFER'    : 0.001,

    # ── Output ─────────────────────────────────────────────────────────────
    'SAVE_PLOTS'         : True,
    'PLOT_DPI'           : 130,
    'BG'                 : '#0d1117',
    'TEAL'               : '#00d4aa',
    'ORANGE'             : '#f7931a',
    'RED'                : '#e74c3c',
    'BLUE'               : '#3498db',
    'GREY'               : '#555555',
    'YELLOW'             : '#f1c40f',
}

print('Config loaded.')
print(f"  Structure TF   : {CFG['STRUCTURE_TF']}")
print(f"  Days           : {CFG['DAYS_TO_LOAD']}")
print(f"  BOS buffer bars: {CFG['BOS_BUFFER_BARS']}")
print(f"  Range threshold: {CFG['RANGE_BARS']} bars")
print(f"  BOS weights    : body={CFG['W_BODY']} range={CFG['W_RANGE']} vol={CFG['W_VOL']}")

## Cell 3 — Data Loader
Loads 1m Binance klines CSV, resamples to structure timeframe.
Computes ATR, volume MA, body ratio, range expansion.

In [ ]:
KLINE_COLS = [
    'open_time','open','high','low','close','volume','close_time',
    'quote_vol','n_trades','taker_buy_base','taker_buy_quote','ignore'
]

def parse_ts(series):
    s = pd.to_numeric(series, errors='coerce').dropna().astype(np.int64)
    unit = 'us' if s.iloc[0] > 1e15 else 'ms'
    return pd.to_datetime(s, unit=unit, utc=True)

def load_klines(path):
    import os
    if not os.path.exists(path):
        raise FileNotFoundError(f'Klines file not found: {path}')
    df = pd.read_csv(path)
    if 'open_time' not in df.columns:
        df.columns = KLINE_COLS
    df = df.dropna(subset=['open_time'])
    df.index = parse_ts(df['open_time'])
    taker_col = 'taker_buy_base' if 'taker_buy_base' in df.columns else 'taker_buy_volume'
    use = ['open','high','low','close','volume', taker_col]
    df = df[[c for c in use if c in df.columns]].apply(pd.to_numeric, errors='coerce')
    if taker_col != 'taker_buy_base':
        df.rename(columns={taker_col: 'taker_buy_base'}, inplace=True)
    df = df[~df.index.duplicated(keep='first')].sort_index()
    print(f'  [1m] {len(df):,} bars | {df.index[0].date()} → {df.index[-1].date()}')
    return df

def clip_days(df, days):
    if df is None: return None
    return df[df.index >= df.index[-1] - pd.Timedelta(days=days)]

def resample_klines(df1m, rule):
    return df1m.resample(rule).agg({
        'open':'first','high':'max','low':'min','close':'last',
        'volume':'sum','taker_buy_base':'sum'
    }).dropna(subset=['open'])

def compute_atr(df, period=14):
    h, l, pc = df['high'], df['low'], df['close'].shift(1)
    tr = pd.concat([h-l, (h-pc).abs(), (l-pc).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1/period, adjust=False).mean()

print('Loading data...')
df1m_raw  = load_klines(CFG['PATH_KLINES_1M'])
df1m_raw  = clip_days(df1m_raw, CFG['DAYS_TO_LOAD'])

df_struct = resample_klines(df1m_raw, CFG['STRUCTURE_TF'])
print(f'  [{CFG["STRUCTURE_TF"]}] {len(df_struct):,} bars')

df_struct['atr']         = compute_atr(df_struct, CFG['ATR_PERIOD'])
df_struct['vol_ma']      = df_struct['volume'].rolling(CFG['VOL_MA_PERIOD']).mean()
df_struct['vol_rel']     = df_struct['volume'] / (df_struct['vol_ma'] + 1e-10)
df_struct['bar_range']   = df_struct['high'] - df_struct['low']
df_struct['body']        = (df_struct['close'] - df_struct['open']).abs()
df_struct['body_ratio']  = df_struct['body'] / (df_struct['bar_range'] + 1e-10)
df_struct['range_exp']   = df_struct['bar_range'] / (df_struct['atr'] + 1e-10)
df_struct['taker_ratio'] = df_struct['taker_buy_base'] / (df_struct['volume'] + 1e-10)
df_struct['log_ret']     = np.log(df_struct['close'] / df_struct['close'].shift(1))
df_struct = df_struct.dropna(subset=['atr', 'vol_ma'])

print(f'Structure bars after warmup: {len(df_struct):,}')
print(f'Date range: {df_struct.index[0].date()} → {df_struct.index[-1].date()}')

## Phase 5A — Swing Detection

**3-bar rule (strict):**
- Swing High at bar[i]: `high[i] > high[i-1] AND high[i] > high[i+1]`
- Swing Low  at bar[i]: `low[i]  < low[i-1]  AND low[i]  < low[i+1]`
- Confirmed at bar[i+1] close — one bar lag, no lookahead.
- All detected swings start as **Minor**. Major promotion happens in Phase 5B.

In [ ]:
def detect_swings(df):
    """
    Returns four Series:
        sh_conf / sl_conf : confirmed swing labels (shifted +1 for one-bar lag)
        sh_raw  / sl_raw  : raw labels at actual pivot bar (for price retrieval)
    """
    highs = df['high'].values
    lows  = df['low'].values
    n     = len(df)

    sh = np.zeros(n, dtype=bool)
    sl = np.zeros(n, dtype=bool)

    for i in range(1, n - 1):
        if highs[i] > highs[i-1] and highs[i] > highs[i+1]:
            sh[i] = True
        if lows[i]  < lows[i-1]  and lows[i]  < lows[i+1]:
            sl[i] = True

    # Shift forward by 1: the swing at bar[i] is known at bar[i+1] close
    sh_conf = np.roll(sh, 1); sh_conf[0] = False
    sl_conf = np.roll(sl, 1); sl_conf[0] = False

    return (
        pd.Series(sh_conf, index=df.index, name='swing_high'),
        pd.Series(sl_conf, index=df.index, name='swing_low'),
        pd.Series(sh,      index=df.index, name='sh_raw'),
        pd.Series(sl,      index=df.index, name='sl_raw'),
    )

print('Detecting swings...')
sh_conf, sl_conf, sh_raw, sl_raw = detect_swings(df_struct)
df_struct['swing_high'] = sh_conf
df_struct['swing_low']  = sl_conf
df_struct['sh_raw']     = sh_raw
df_struct['sl_raw']     = sl_raw

n_sh = sh_conf.sum()
n_sl = sl_conf.sum()
print(f'  Raw swing highs: {n_sh:,}')
print(f'  Raw swing lows : {n_sl:,}')
print(f'  Total swings   : {n_sh + n_sl:,}')
print(f'  Avg bars/swing : {len(df_struct) / (n_sh + n_sl):.1f}')

## Phase 5B — Major/Minor Classification & BOS Detection

**State machine (walks forward bar by bar):**

1. **BOS Trigger**: close crosses the current Major Swing level.
2. **Buffer check**: look ahead `BOS_BUFFER_BARS` bars.
   - If price returns through the broken level within the buffer → `FAILED_BOS`. Level stays, leg continues.
   - If price holds → `CONFIRMED_BOS`. Transition state, promote major levels.
3. **On confirmed bullish BOS**:
   - Retroactively find the **lowest swing low** in the leg (since last major SL) → promote to Major SL.
   - The **first confirmed swing high that forms after the BOS bar** → promote to new Major SH.
4. **Scoring** (all BOS triggers, confirmed or failed):
   - `close_type` : BODY / CLOSE / WICK (based on close/open vs level)
   - `wick_score` : wick extension beyond level normalised by ATR
   - `disp_score` : weighted body_ratio + range_exp + vol_rel
   - `total_score`: disp_score + close_bonus
5. **Range tag**: if bars since last confirmed BOS > `RANGE_BARS`, tag bar as RANGING.

In [ ]:
def classify_structure(df, cfg):
    """
    Walk forward bar by bar.

    Returns
    -------
    bos_records   : list[dict]  — one entry per BOS trigger (confirmed or failed)
    swing_records : list[dict]  — one entry per confirmed swing (with major flag)
    """

    # ── Raw arrays ────────────────────────────────────────────────────────
    highs      = df['high'].values
    lows       = df['low'].values
    opens      = df['open'].values
    closes     = df['close'].values
    sh_conf    = df['swing_high'].values   # True at bar[i+1] when bar[i] is a swing high
    sl_conf    = df['swing_low'].values
    sh_raw     = df['sh_raw'].values       # True at the actual pivot bar
    sl_raw     = df['sl_raw'].values
    body_ratio = df['body_ratio'].values
    range_exp  = df['range_exp'].values
    vol_rel    = df['vol_rel'].values
    taker_r    = df['taker_ratio'].values
    atr        = df['atr'].values
    idx        = df.index
    n          = len(df)

    buffer_bars = cfg['BOS_BUFFER_BARS']

    # ── State ─────────────────────────────────────────────────────────────
    trend          = 'UNDEFINED'   # BULL / BEAR / UNDEFINED
    major_sh_price = None          # price of current active Major Swing High
    major_sh_bar   = None          # bar index of current active Major Swing High
    major_sl_price = None          # price of current active Major Swing Low
    major_sl_bar   = None          # bar index of current active Major Swing Low

    # Leg tracking: all swing lows/highs since the last confirmed BOS
    # Each entry: (price, bar_index)
    leg_sl = []   # swing lows accumulated in current leg
    leg_sh = []   # swing highs accumulated in current leg

    # After a confirmed BOS, we wait for the first new confirmed swing high (bull)
    # or swing low (bear) to promote it as the new Major level.
    # These flags track that waiting state.
    waiting_for_major_sh = False   # set True after bullish BOS
    waiting_for_major_sl = False   # set True after bearish BOS

    bars_since_bos = 0   # for range detection

    bos_records   = []
    swing_records = []

    # ── BOS Scorer ────────────────────────────────────────────────────────
    def score_bos(bar_i, level_price, direction):
        """
        direction: +1 bullish BOS (close broke above level)
                   -1 bearish BOS (close broke below level)

        close_type categories:
            BODY  : open AND close are both beyond the level (full body break)
            CLOSE : only the close is beyond the level
            WICK  : close did NOT exceed the level — only the wick did
                    (This shouldn't trigger since we use close-based trigger,
                     but kept for completeness / edge cases.)

        wick_score: how far the extreme wick went beyond the level,
                    normalised by ATR. Captures wick-based strength
                    independent of close_type.
        """
        br  = float(body_ratio[bar_i])
        re  = float(range_exp[bar_i])
        vr  = float(vol_rel[bar_i])
        tk  = float(taker_r[bar_i])
        o   = opens[bar_i]
        c   = closes[bar_i]
        h   = highs[bar_i]
        lo  = lows[bar_i]
        at  = float(atr[bar_i]) if atr[bar_i] > 0 else 1.0

        # ── Displacement score (0-1) ──────────────────────────────────────
        body_score  = br
        range_score = min(re / cfg['RANGE_EXP_CAP'], 1.0)
        vol_score   = min(vr / cfg['VOL_REL_CAP'],   1.0)
        disp_score  = (body_score  * cfg['W_BODY'] +
                       range_score * cfg['W_RANGE'] +
                       vol_score   * cfg['W_VOL'])

        # ── Close type ────────────────────────────────────────────────────
        if direction == 1:
            body_broke  = (o > level_price) and (c > level_price)
            close_broke = c > level_price
            wick_ext    = max(h - level_price, 0.0)   # how far wick went above
        else:
            body_broke  = (o < level_price) and (c < level_price)
            close_broke = c < level_price
            wick_ext    = max(level_price - lo, 0.0)  # how far wick went below

        if body_broke:
            close_bonus = cfg['CLOSE_BONUS_BODY']
            close_type  = 'BODY'
        elif close_broke:
            close_bonus = cfg['CLOSE_BONUS_CLOSE']
            close_type  = 'CLOSE'
        else:
            close_bonus = cfg['CLOSE_BONUS_WICK']
            close_type  = 'WICK'

        wick_score  = min(wick_ext / at, 2.0)   # cap at 2× ATR
        total_score = min(disp_score + close_bonus, 1.0)

        return {
            'body_score'   : body_score,
            'range_score'  : range_score,
            'vol_score'    : vol_score,
            'disp_score'   : disp_score,
            'wick_score'   : wick_score,
            'close_type'   : close_type,
            'close_bonus'  : close_bonus,
            'total_score'  : total_score,
            'taker_ratio'  : tk,
            'vol_rel'      : vr,
            'bar_range_atr': re,
        }

    # ── Buffer check ──────────────────────────────────────────────────────
    def bos_holds(bar_i, level_price, direction):
        """
        Look ahead buffer_bars bars after the BOS trigger bar.
        Returns True if price does NOT return through the level.
        Uses close for the return check (consistent with the trigger).
        """
        end = min(bar_i + buffer_bars + 1, n)
        for j in range(bar_i + 1, end):
            if direction == 1:
                # Bull BOS: price must not close back below the level
                if closes[j] < level_price:
                    return False
            else:
                # Bear BOS: price must not close back above the level
                if closes[j] > level_price:
                    return False
        return True

    # ── Main loop ─────────────────────────────────────────────────────────
    for i in range(n):
        bars_since_bos += 1
        is_ranging = bars_since_bos > cfg['RANGE_BARS']

        # ── Step 1: Register newly confirmed swings into leg accumulators
        # sh_conf[i] is True when bar[i-1] was a swing high (confirmed at bar[i] close)
        # So the actual swing high price is highs[i-1]
        if sh_conf[i] and i >= 1:
            sh_price = highs[i-1]
            sh_bar   = i - 1
            leg_sh.append((sh_price, sh_bar))
            swing_records.append({
                'ts'      : idx[sh_bar],
                'type'    : 'HIGH',
                'price'   : sh_price,
                'major'   : False,
                'bar_idx' : sh_bar,
            })

            # ── If we're waiting for the first swing high after a bull BOS,
            #    this becomes the new Major Swing High
            if waiting_for_major_sh:
                major_sh_price       = sh_price
                major_sh_bar         = sh_bar
                swing_records[-1]['major'] = True   # promote it
                waiting_for_major_sh = False

        if sl_conf[i] and i >= 1:
            sl_price = lows[i-1]
            sl_bar   = i - 1
            leg_sl.append((sl_price, sl_bar))
            swing_records.append({
                'ts'      : idx[sl_bar],
                'type'    : 'LOW',
                'price'   : sl_price,
                'major'   : False,
                'bar_idx' : sl_bar,
            })

            # ── If we're waiting for the first swing low after a bear BOS,
            #    this becomes the new Major Swing Low
            if waiting_for_major_sl:
                major_sl_price       = sl_price
                major_sl_bar         = sl_bar
                swing_records[-1]['major'] = True
                waiting_for_major_sl = False

        # ── Step 2: Bootstrap — need at least one major SH and SL to watch
        # On the first swing high seen, set it as initial major SH candidate.
        # On the first swing low seen, set it as initial major SL candidate.
        # This bootstraps the state machine without assuming a direction.
        if major_sh_price is None and leg_sh:
            major_sh_price, major_sh_bar = leg_sh[-1]
            # Mark it as major in swing_records
            for rec in swing_records:
                if rec['bar_idx'] == major_sh_bar and rec['type'] == 'HIGH':
                    rec['major'] = True
                    break

        if major_sl_price is None and leg_sl:
            major_sl_price, major_sl_bar = leg_sl[-1]
            for rec in swing_records:
                if rec['bar_idx'] == major_sl_bar and rec['type'] == 'LOW':
                    rec['major'] = True
                    break

        # Need both major levels set before we can detect BOS
        if major_sh_price is None or major_sl_price is None:
            continue

        # ── Step 3: BOS detection — close-based trigger ───────────────────

        # ── BULLISH BOS: close above Major Swing High
        if closes[i] > major_sh_price:
            scores    = score_bos(i, major_sh_price, +1)
            confirmed = bos_holds(i, major_sh_price, +1)
            status    = 'CONFIRMED' if confirmed else 'FAILED'

            # Retroactively find the lowest swing low in the current leg
            # (from major_sl_bar to now) → this becomes the new Major SL on confirm
            new_major_sl_price = None
            new_major_sl_bar   = None
            if leg_sl:
                new_major_sl_price, new_major_sl_bar = min(leg_sl, key=lambda x: x[0])

            bos_records.append({
                'ts'             : idx[i],
                'bar_idx'        : i,
                'direction'      : 'BULL',
                'broken_level'   : major_sh_price,
                'broken_bar_idx' : major_sh_bar,
                'new_major_sl'   : new_major_sl_price,
                'new_major_sl_bar': new_major_sl_bar,
                'status'         : status,
                'is_ranging'     : is_ranging,
                'prev_trend'     : trend,
                **scores,
            })

            if confirmed:
                # ── Promote lowest swing low of the leg to Major SL
                if new_major_sl_price is not None:
                    for rec in swing_records:
                        if rec['bar_idx'] == new_major_sl_bar and rec['type'] == 'LOW':
                            rec['major'] = True
                            break
                    major_sl_price = new_major_sl_price
                    major_sl_bar   = new_major_sl_bar

                # ── New Major SH: first swing high that forms AFTER this bar
                #    Set a flag; the swing registration block above will catch it.
                waiting_for_major_sh = True
                # Temporarily advance major_sh to current bar's high so we
                # don't re-trigger BOS on the same level until a new one is set.
                # (Will be overwritten when waiting_for_major_sh resolves.)
                major_sh_price = highs[i]
                major_sh_bar   = i

                trend          = 'BULL'
                bars_since_bos = 0

                # Reset leg — start fresh from current major SL
                leg_sl = [(major_sl_price, major_sl_bar)] if major_sl_price else []
                leg_sh = []

            # FAILED BOS: log it, keep everything intact, leg continues
            # (no state changes, no leg reset)

        # ── BEARISH BOS: close below Major Swing Low
        elif closes[i] < major_sl_price:
            scores    = score_bos(i, major_sl_price, -1)
            confirmed = bos_holds(i, major_sl_price, -1)
            status    = 'CONFIRMED' if confirmed else 'FAILED'

            # Retroactively find the highest swing high in the current leg
            new_major_sh_price = None
            new_major_sh_bar   = None
            if leg_sh:
                new_major_sh_price, new_major_sh_bar = max(leg_sh, key=lambda x: x[0])

            bos_records.append({
                'ts'             : idx[i],
                'bar_idx'        : i,
                'direction'      : 'BEAR',
                'broken_level'   : major_sl_price,
                'broken_bar_idx' : major_sl_bar,
                'new_major_sh'   : new_major_sh_price,
                'new_major_sh_bar': new_major_sh_bar,
                'status'         : status,
                'is_ranging'     : is_ranging,
                'prev_trend'     : trend,
                **scores,
            })

            if confirmed:
                # ── Promote highest swing high of the leg to Major SH
                if new_major_sh_price is not None:
                    for rec in swing_records:
                        if rec['bar_idx'] == new_major_sh_bar and rec['type'] == 'HIGH':
                            rec['major'] = True
                            break
                    major_sh_price = new_major_sh_price
                    major_sh_bar   = new_major_sh_bar

                # ── New Major SL: first swing low that forms AFTER this bar
                waiting_for_major_sl = True
                major_sl_price = lows[i]
                major_sl_bar   = i

                trend          = 'BEAR'
                bars_since_bos = 0

                leg_sh = [(major_sh_price, major_sh_bar)] if major_sh_price else []
                leg_sl = []

    return bos_records, swing_records


print('Classifying structure (Major/Minor + BOS)...')
bos_records, swing_records = classify_structure(df_struct, CFG)

bos_df   = pd.DataFrame(bos_records)
swing_df = pd.DataFrame(swing_records)

if len(bos_df) > 0:
    bos_df = bos_df.set_index('ts')
    conf   = bos_df['status'] == 'CONFIRMED'
    failed = bos_df['status'] == 'FAILED'
    print(f'\nBOS triggers total    : {len(bos_df):,}')
    print(f'  Confirmed           : {conf.sum():,}')
    print(f'  Failed (returned)   : {failed.sum():,}')
    print(f'  BULL confirmed      : {((bos_df["direction"]=="BULL") & conf).sum():,}')
    print(f'  BEAR confirmed      : {((bos_df["direction"]=="BEAR") & conf).sum():,}')
    print(f'  Avg total_score     : {bos_df["total_score"].mean():.3f}')
    print(f'  Avg score (confirmed): {bos_df.loc[conf, "total_score"].mean():.3f}')
    print(f'  Avg score (failed)   : {bos_df.loc[failed, "total_score"].mean():.3f}')

if len(swing_df) > 0:
    major_count = swing_df['major'].sum()
    print(f'\nSwing levels total    : {len(swing_df):,}')
    print(f'  Major               : {major_count:,}')
    print(f'  Minor               : {len(swing_df) - major_count:,}')

## Phase 5C — BOS Outcome Measurement
For each **confirmed** BOS, measure whether it was genuine or fake.
- Genuine = price stayed on the correct side for `BOS_GENUINE_HOLD` bars
- Fake    = price returned through the broken level within the hold window
- Forward returns recorded at all configured horizons

Note: FAILED_BOS events are excluded from outcome measurement (they already failed the buffer test).

In [ ]:
if len(bos_df) == 0:
    print('No BOS events found.')
else:
    closes_s  = df_struct['close'].values
    highs_s   = df_struct['high'].values
    lows_s    = df_struct['low'].values
    hold_bars = CFG['BOS_GENUINE_HOLD']
    buffer    = CFG['BOS_HOLD_BUFFER']

    outcomes = []

    for ts, row in bos_df.iterrows():
        # Only measure confirmed BOS events
        if row['status'] != 'CONFIRMED':
            outcomes.append({'genuine': np.nan})
            continue

        bi        = int(row['bar_idx'])
        level     = float(row['broken_level'])
        direction = row['direction']

        if bi >= len(closes_s) - max(CFG['BOS_FWD_BARS']) - 1:
            outcomes.append({'genuine': np.nan})
            continue

        entry_px = closes_s[bi]

        # Genuine test: price must not return through broken level
        genuine = True
        for j in range(bi + 1, min(bi + hold_bars + 1, len(closes_s))):
            if direction == 'BULL':
                if lows_s[j] < level * (1 - buffer):
                    genuine = False; break
            else:
                if highs_s[j] > level * (1 + buffer):
                    genuine = False; break

        # Forward returns
        dir_sign = 1 if direction == 'BULL' else -1
        fwd_rets = {}
        for fwd in CFG['BOS_FWD_BARS']:
            target_bar      = min(bi + fwd, len(closes_s) - 1)
            fwd_rets[f'fwd_{fwd}'] = dir_sign * (closes_s[target_bar] / entry_px - 1) * 100

        outcomes.append({'genuine': genuine, **fwd_rets})

    outcome_df = pd.DataFrame(outcomes, index=bos_df.index)
    bos_df     = pd.concat([bos_df, outcome_df], axis=1)

    confirmed_df   = bos_df[bos_df['status'] == 'CONFIRMED']
    valid_outcomes = confirmed_df[confirmed_df['genuine'].notna()]
    genuine_mask   = valid_outcomes['genuine'] == True
    fake_mask      = valid_outcomes['genuine'] == False

    print(f'Confirmed BOS outcomes: {len(valid_outcomes):,}')
    print(f'  Genuine : {genuine_mask.sum():,} ({genuine_mask.mean()*100:.1f}%)')
    print(f'  Fake    : {fake_mask.sum():,} ({fake_mask.mean()*100:.1f}%)')
    print()

    print('Mean forward return: genuine vs fake (confirmed BOS only)')
    for fwd in CFG['BOS_FWD_BARS']:
        col = f'fwd_{fwd}'
        if col not in bos_df.columns: continue
        g_mean = valid_outcomes[genuine_mask][col].mean()
        f_mean = valid_outcomes[fake_mask][col].mean()
        print(f'  {fwd:>3} bars: genuine={g_mean:+.4f}%  fake={f_mean:+.4f}%')

    print()
    print('Score comparison: genuine vs fake confirmed BOS')
    score_cols = ['body_score','range_score','vol_score','disp_score',
                  'wick_score','total_score']
    for col in score_cols:
        if col not in bos_df.columns: continue
        g = valid_outcomes[genuine_mask][col].dropna()
        f = valid_outcomes[fake_mask][col].dropna()
        if len(g) > 3 and len(f) > 3:
            stat, p = mannwhitneyu(g, f, alternative='greater')
            print(f'  {col:<15}: genuine={g.mean():.3f}  fake={f.mean():.3f}  '
                  f'p={p:.4f} {"*" if p < 0.05 else ""}')

## Phase 5D — Score Parameter Analysis
Spearman correlations and genuine rate breakdown by close type.
Purpose: identify which scoring parameter most reliably predicts a genuine BOS.

In [ ]:
if len(bos_df) == 0:
    print('No BOS events.')
else:
    confirmed_df   = bos_df[bos_df['status'] == 'CONFIRMED']
    valid_outcomes = confirmed_df[confirmed_df['genuine'].notna()].copy()
    valid_outcomes['genuine_int'] = valid_outcomes['genuine'].astype(int)

    print('=' * 60)
    print('SPEARMAN ρ — score components vs genuine outcome')
    print('(confirmed BOS only)')
    print('=' * 60)

    spearman_results = {}
    score_cols = ['body_score','range_score','vol_score','disp_score',
                  'wick_score','close_bonus','total_score']
    for col in score_cols:
        if col not in valid_outcomes.columns: continue
        vals = valid_outcomes[col].dropna()
        gen  = valid_outcomes.loc[vals.index, 'genuine_int']
        if len(vals) > 5:
            rho, pv = spearmanr(vals, gen)
            spearman_results[col] = (rho, pv)
            sig = '  ← SIGNIFICANT' if pv < 0.05 else ''
            print(f'  {col:<15}: ρ={rho:+.4f}  p={pv:.4f}{sig}')

    print()
    print('GENUINE RATE BY CLOSE TYPE (confirmed BOS only)')
    print('-' * 40)
    if 'close_type' in valid_outcomes.columns:
        for ct in ['BODY', 'CLOSE', 'WICK']:
            sub = valid_outcomes[valid_outcomes['close_type'] == ct]
            if len(sub) >= 2:
                rate = sub['genuine'].mean() * 100
                print(f'  {ct:<6}: {rate:.1f}%  (n={len(sub)})')

    print()
    print('CONFIRMED vs FAILED BOS — score comparison')
    print('-' * 40)
    conf_df   = bos_df[bos_df['status'] == 'CONFIRMED']
    failed_df = bos_df[bos_df['status'] == 'FAILED']
    for col in ['body_score','range_score','vol_score','wick_score','total_score']:
        if col not in bos_df.columns: continue
        c_mean = conf_df[col].mean()
        f_mean = failed_df[col].mean() if len(failed_df) > 0 else np.nan
        print(f'  {col:<15}: confirmed={c_mean:.3f}  failed={f_mean:.3f}')

    print()
    print('RANGING vs TRENDING BOS — genuine rate')
    print('-' * 40)
    if 'is_ranging' in valid_outcomes.columns:
        for r_flag, label in [(True, 'Ranging'), (False, 'Trending')]:
            sub = valid_outcomes[valid_outcomes['is_ranging'] == r_flag]
            if len(sub) >= 2:
                rate = sub['genuine'].mean() * 100
                print(f'  {label:<10}: genuine={rate:.1f}%  (n={len(sub)})')

## Phase 5E — Visualisation
Chart 1: Price with confirmed/failed BOS markers and Major swing levels.
Chart 2: Score distributions by close type.
Chart 3: Genuine rate by close type and score bucket.

In [ ]:
BG     = CFG['BG']
TEAL   = CFG['TEAL']
ORANGE = CFG['ORANGE']
RED    = CFG['RED']
BLUE   = CFG['BLUE']
GREY   = CFG['GREY']
YELLOW = CFG['YELLOW']

# ── Chart 1: Price + BOS markers + Major swing levels ─────────────────────
fig, ax = plt.subplots(figsize=(18, 7), facecolor=BG)
ax.set_facecolor(BG)

prices = df_struct['close'].values
x_idx  = np.arange(len(df_struct))
ax.plot(x_idx, prices, color=GREY, lw=0.8, zorder=1)

# Plot Major swing levels
if len(swing_df) > 0:
    major_df = swing_df[swing_df['major']]
    for _, row in major_df.iterrows():
        bi    = row['bar_idx']
        price = row['price']
        color = TEAL if row['type'] == 'LOW' else ORANGE
        ax.scatter(bi, price, color=color, s=40, zorder=3)
        ax.axhline(price, color=color, lw=0.4, ls='--', alpha=0.3)

# Plot BOS events
if len(bos_df) > 0:
    for ts, row in bos_df.iterrows():
        bi    = int(row['bar_idx'])
        price = prices[bi]
        if row['status'] == 'CONFIRMED':
            color  = TEAL if row['direction'] == 'BULL' else RED
            marker = '^' if row['direction'] == 'BULL' else 'v'
            ax.scatter(bi, price, color=color, s=80, marker=marker, zorder=4)
        else:  # FAILED
            ax.scatter(bi, price, color=YELLOW, s=50, marker='x', zorder=4)

legend_items = [
    mpatches.Patch(color=TEAL,   label='Bull BOS confirmed / Major SL'),
    mpatches.Patch(color=RED,    label='Bear BOS confirmed'),
    mpatches.Patch(color=ORANGE, label='Major SH'),
    mpatches.Patch(color=YELLOW, label='Failed BOS'),
]
ax.legend(handles=legend_items, facecolor=BG, edgecolor=GREY,
          labelcolor='white', fontsize=8)
ax.set_title('Price | BOS Events | Major Swing Levels', color='white', fontsize=12)
ax.tick_params(colors='white')
ax.spines[['top','right']].set_visible(False)
for sp in ['bottom','left']: ax.spines[sp].set_color(GREY)

plt.tight_layout()
if CFG['SAVE_PLOTS']:
    plt.savefig('phase5_price_bos.png', dpi=CFG['PLOT_DPI'],
                bbox_inches='tight', facecolor=BG)
plt.show()

# ── Chart 2: Score distributions by close type ────────────────────────────
if len(bos_df) > 0 and 'close_type' in bos_df.columns:
    conf_df = bos_df[bos_df['status'] == 'CONFIRMED'].copy()
    score_cols_plot = ['body_score','range_score','vol_score','wick_score','total_score']
    score_cols_plot = [c for c in score_cols_plot if c in conf_df.columns]

    fig, axes = plt.subplots(1, len(score_cols_plot), figsize=(4*len(score_cols_plot), 5),
                             facecolor=BG)
    ct_colors = {'BODY': TEAL, 'CLOSE': ORANGE, 'WICK': GREY}

    for ax, col in zip(axes, score_cols_plot):
        ax.set_facecolor(BG)
        for ct, color in ct_colors.items():
            sub = conf_df[conf_df['close_type'] == ct][col].dropna()
            if len(sub) > 2:
                ax.hist(sub, bins=10, alpha=0.6, color=color, label=ct,
                        density=True)
        ax.set_title(col, color='white', fontsize=9)
        ax.tick_params(colors='white', labelsize=7)
        ax.spines[['top','right']].set_visible(False)
        for sp in ['bottom','left']: ax.spines[sp].set_color(GREY)
        ax.legend(fontsize=7, facecolor=BG, labelcolor='white')

    plt.suptitle('Score Distributions by Close Type (Confirmed BOS)',
                 color='white', fontsize=11)
    plt.tight_layout()
    if CFG['SAVE_PLOTS']:
        plt.savefig('phase5_score_dist.png', dpi=CFG['PLOT_DPI'],
                    bbox_inches='tight', facecolor=BG)
    plt.show()

# ── Chart 3: Genuine rate by total_score decile ───────────────────────────
if len(bos_df) > 0:
    conf_df = bos_df[(bos_df['status']=='CONFIRMED') & bos_df['genuine'].notna()].copy()
    if len(conf_df) >= 10:
        conf_df['score_decile'] = pd.qcut(conf_df['total_score'], q=5,
                                           labels=['Q1','Q2','Q3','Q4','Q5'],
                                           duplicates='drop')
        gen_by_decile = conf_df.groupby('score_decile')['genuine'].mean() * 100
        n_by_decile   = conf_df.groupby('score_decile').size()

        fig, ax = plt.subplots(figsize=(8, 4), facecolor=BG)
        ax.set_facecolor(BG)
        bars = ax.bar(gen_by_decile.index.astype(str), gen_by_decile.values,
                      color=TEAL, width=0.5)
        for bar, (dec, n) in zip(bars, n_by_decile.items()):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'n={n}', ha='center', color='white', fontsize=8)
        ax.axhline(50, color=GREY, lw=0.8, ls='--')
        ax.set_ylabel('Genuine BOS %', color='white')
        ax.set_xlabel('Total Score Quintile (Q1=lowest)', color='white')
        ax.set_title('Genuine Rate by Score Quintile (Confirmed BOS)',
                     color='white', fontsize=11)
        ax.tick_params(colors='white')
        ax.spines[['top','right']].set_visible(False)
        for sp in ['bottom','left']: ax.spines[sp].set_color(GREY)
        plt.tight_layout()
        if CFG['SAVE_PLOTS']:
            plt.savefig('phase5_genuine_by_score.png', dpi=CFG['PLOT_DPI'],
                        bbox_inches='tight', facecolor=BG)
        plt.show()

## Summary
Full findings printout. Use these numbers to update score weights and thresholds.

In [ ]:
sep = '=' * 70
print(sep)
print('  SWING STRUCTURE & BOS ANALYSIS — SUMMARY v2')
print(sep)

if len(bos_df) > 0:
    confirmed_df   = bos_df[bos_df['status'] == 'CONFIRMED']
    failed_df      = bos_df[bos_df['status'] == 'FAILED']
    valid_outcomes = confirmed_df[confirmed_df['genuine'].notna()]

    print(f'\nDATA')
    print(f'  Timeframe    : {CFG["STRUCTURE_TF"]}')
    print(f'  Period       : {df_struct.index[0].date()} → {df_struct.index[-1].date()}')
    print(f'  Total bars   : {len(df_struct):,}')

    print(f'\nSWING LEVELS')
    if len(swing_df) > 0:
        print(f'  Total        : {len(swing_df):,}')
        print(f'  Major        : {swing_df["major"].sum():,}')
        print(f'  Minor        : {(~swing_df["major"]).sum():,}')

    print(f'\nBOS EVENTS')
    print(f'  Total triggers  : {len(bos_df):,}')
    print(f'  Confirmed       : {len(confirmed_df):,}')
    print(f'  Failed (buffer) : {len(failed_df):,}')
    if len(valid_outcomes) > 0:
        g = (valid_outcomes["genuine"] == True).sum()
        f = (valid_outcomes["genuine"] == False).sum()
        print(f'  Genuine         : {g:,} ({g/len(valid_outcomes)*100:.1f}%)')
        print(f'  Fake            : {f:,} ({f/len(valid_outcomes)*100:.1f}%)')

    print(f'\nSCORE PREDICTIVENESS (Spearman ρ vs genuine outcome)')
    score_cols = ['body_score','range_score','vol_score','disp_score',
                  'wick_score','close_bonus','total_score']
    if len(valid_outcomes) > 5:
        valid_outcomes_copy = valid_outcomes.copy()
        valid_outcomes_copy['genuine_int'] = valid_outcomes_copy['genuine'].astype(int)
        for col in score_cols:
            if col not in valid_outcomes_copy.columns: continue
            vals = valid_outcomes_copy[col].dropna()
            gen  = valid_outcomes_copy.loc[vals.index, 'genuine_int']
            if len(vals) > 5:
                rho, pv = spearmanr(vals, gen)
                sig = ' *' if pv < 0.05 else ''
                print(f'  {col:<15}: ρ={rho:+.4f}  p={pv:.4f}{sig}')

    print(f'\nGENUINE RATE BY CLOSE TYPE')
    if 'close_type' in valid_outcomes.columns:
        for ct in ['BODY', 'CLOSE', 'WICK']:
            sub = valid_outcomes[valid_outcomes['close_type'] == ct]
            if len(sub) >= 2:
                print(f'  {ct:<6}: {sub["genuine"].mean()*100:.1f}%  (n={len(sub)})')

    print(f'\nFORWARD RETURNS AT GENUINE BOS')
    gen_only = valid_outcomes[valid_outcomes['genuine'] == True]
    for fwd in CFG['BOS_FWD_BARS']:
        col = f'fwd_{fwd}'
        if col in gen_only.columns:
            m = gen_only[col].mean()
            w = (gen_only[col] > 0).mean() * 100
            print(f'  {fwd:>3} bars: mean={m:+.4f}%  win%={w:.1f}%')

print()
print(sep)
print('  Interpretation guide:')
print('  1. Which score col has highest Spearman ρ and p < 0.05?')
print('     → That parameter is most predictive of a genuine BOS.')
print('  2. BODY close type has highest genuine %?')
print('     → Require full body break as minimum BOS filter.')
print('  3. Confirmed << Total triggers?')
print('     → Increase BOS_BUFFER_BARS or check data quality.')
print('  4. Run on multiple timeframes and assets to generalise.')
print(sep)